In [1]:
import json
from typing import List
import numpy as np
import librosa
import sys
sys.path.append('/home/u1501463/tool_use_LALM/')

def load_audio(path: str, sr: int = 16000):
    y, _ = librosa.load(path, sr=sr, mono=True)
    return y.astype(np.float32), sr

def load_subset(subset_path: str) -> List[dict]:
    with open(subset_path, "r", encoding="utf-8") as f:
        subset = json.load(f)
    if not isinstance(subset, list):
        raise ValueError("Subset file must contain a JSON list.")
    return subset

subset_path = '/home/u1501463/tool_use_LALM/dcase_subset.json'

subset = load_subset(subset_path)
import random
random.shuffle(subset)

In [2]:
from tools import generate_tool_descriptions
from tools import (
    ClippingTool,
    DenoiseTool,
    AmplitudeNormalizeTool,
    LoudnessNormalizeTool,
    DCOffsetRemovalTool,
    SpectralNormalizeTool,
    TrimSilenceTool,
    PreEmphasisTool,
    SourceSeparationTool,
)

tool_set =     [
        ClippingTool,
        DenoiseTool,
        # AmplitudeNormalizeTool,
        # LoudnessNormalizeTool,
        # DCOffsetRemovalTool,
        # SpectralNormalizeTool,
        # TrimSilenceTool,
        # PreEmphasisTool,
        SourceSeparationTool,
    ]

tool_set = None

tools_description = generate_tool_descriptions(
    tool_set
)

print(tools_description)


asr: Transcribe speech from a WAV audio segment into text. Requires audio_path plus HH:MM:SS.mmm timestamps for audio_begin and audio_end, with optional language selection.
Parameters:
- audio_begin (string), required, format=HH:MM:SS.mmm
- audio_end (string), required, format=HH:MM:SS.mmm
- audio_path (string), required
- language (string), optional

clipping: Extract a WAV audio segment from a source file between specified HH:MM:SS.mmm begin and end timestamps.
Parameters:
- audio_begin (string), required, format=HH:MM:SS.mmm
- audio_end (string), required, format=HH:MM:SS.mmm
- audio_path (string), required

denoise: Apply noise reduction or echo cancellation to a WAV audio segment using the chosen algorithm. Requires audio_path, audio_begin, audio_end, and algorithm; optional noise_factor, sensitivity, and output_path control processing and output destination.
Parameters:
- algorithm (string), required, allowed=['spectral_subtraction', 'wiener', 'echo_cancellation', 'adaptive']
- a

In [ ]:
from tqdm import tqdm
from gemini.gemini import gemini_inference
from shutil import copyfile

from prompts import QA_prompt, gen_tool_prompt

# tools_description = generate_tool_descriptions()

audio_token = "<AUDIO_TOKEN>"  # Placeholder for the actual audio token to be used in the prompt

audio_target_dir = '/home/u1501463/tool_use_LALM/dcase_audios'


tool_schedules = []

tool_cnt = {}

output_path = './tool_schedules.json'

for item in tqdm(subset):
    question = item.get("question", "")
    choices = item.get("choice", [])
    answer = item.get("answer", "")
    audio_path = item.get("audio_path", "")

    # Copy the audio file to the target directory
    copyfile(audio_path, f"{audio_target_dir}/{item.get('id', 'default')}.wav")

    gen_tool_oracle_prompt, user_state = QA_prompt(
        question=question,
        options = choices,
        tools_description=tools_description,
        audio_token = audio_token,
        tool_results = None
    )

    gen_tool_oracle_prompt = gen_tool_oracle_prompt + user_state

    # print(gen_tool_oracle_prompt)

    # break
    # print(question)
    # print(choices)
    # print(answer)
    # print(audio_path)
    output = gemini_inference(
        instruction=gen_tool_oracle_prompt,
        audio_token=audio_token,
        audio_path=audio_path,
        thinking_level='HIGH',
        response_mime_type="application/json",
        # response_schema=tool_schema
    )



    # tool_schedule = json.loads(output.text)
    print(question)
    print(choices)
    print(answer)
    print(output.text)
    # # generated_tool_oracle.append({
    # #     'problem_data': item,
    # #     'generated_tool_oracle': output.text
    # # })

    # for tool in tool_schedule:
    #     tool_name = tool.get("tool", "UnknownTool")
    #     tool_cnt[tool_name] = tool_cnt.get(tool_name, 0) + 1

    # print('\r', tool_cnt)
    # tool_schedules.append((item, tool_schedule))
    # with open(output_path, "w") as f:
    #     json.dump(tool_schedules, f, indent=4)
    break

  0%|          | 0/90 [00:00<?, ?it/s]/home/u1501463/miniconda3/envs/gemini/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
print(gen_tool_oracle_prompt)


You are an expert in audio analysis and question answering.

You are given a question about an audio clip along with multiple-choice options.
You have access to a set of tools that can extract information from the audio.

asr: Transcribe speech from a WAV audio segment into text. Requires audio_path plus HH:MM:SS.mmm timestamps for audio_begin and audio_end, with optional language selection.
Parameters:
- audio_begin (string), required, format=HH:MM:SS.mmm
- audio_end (string), required, format=HH:MM:SS.mmm
- audio_path (string), required
- language (string), optional

clipping: Extract a WAV audio segment from a source file between specified HH:MM:SS.mmm begin and end timestamps.
Parameters:
- audio_begin (string), required, format=HH:MM:SS.mmm
- audio_end (string), required, format=HH:MM:SS.mmm
- audio_path (string), required

denoise: Apply noise reduction or echo cancellation to a WAV audio segment using the chosen algorithm. Requires audio_path, audio_begin, audio_end, and algori